<a href="https://colab.research.google.com/github/xbtnaue-cloud/xongba-tong/blob/main/batong.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, to_timestamp, sum
from google.colab import drive
drive.mount('/content/drive')

# Initialize Spark Session
spark = SparkSession.builder.appName("DataLakePipeline").getOrCreate()

# --- CHUẨN BỊ ĐƯỜNG DẪN ---
datalake_path = '/content/drive/MyDrive/DataLake_Lab'
bronze_path = f'{datalake_path}/Raw_Zone/sales_data.csv'
silver_path = f'{datalake_path}/Silver_Zone/sales_cleaned'
gold_path = f'{datalake_path}/Gold_Zone/country_revenue_summary'

# ==========================================
# CÂU 1: TẠO TẦNG SILVER (LÀM SẠCH CHUYÊN SÂU)
# ==========================================
print("⏳ 1. Đang xử lý tầng Silver...")
df_bronze = spark.read.csv(bronze_path, header=True, inferSchema=True)

# Yêu cầu:
# 1. Bỏ CustomerID bị Null
# 2. Bỏ các đơn hàng bị hủy (InvoiceNo bắt đầu bằng chữ 'C')
# 3. Ép kiểu thời gian và tạo cột 'Year'
df_silver = df_bronze.filter(col("CustomerID").isNotNull()) \
                     .filter(~col("InvoiceNo").startswith("C")) \
                     .withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm")) \
                     .withColumn("Year", year(col("InvoiceDate"))) \
                     .withColumn("TotalAmount", col("Quantity") * col("UnitPrice"))

# Lưu xuống Drive (Không cần phân vùng ở tầng này)
df_silver.write.mode("overwrite").parquet(silver_path)


# ==========================================
# CÂU 2: TẠO TẦNG GOLD (TỔNG HỢP & PHÂN VÙNG)
# ==========================================
print("⏳ 2. Đang xử lý tầng Gold...")
# Đọc lại dữ liệu chuẩn từ tầng Silver
df_clean = spark.read.parquet(silver_path)

# Yêu cầu: Tính tổng doanh thu (TotalAmount) theo từng Quốc gia (Country) và Năm (Year)
df_gold = df_clean.groupBy("Country", "Year") \
                  .agg(sum("TotalAmount").alias("TotalRevenue"))

# Yêu cầu: Lưu dữ liệu tầng Gold ra Drive, PHÂN VÙNG theo 'Year'
df_gold.write \
       .partitionBy("Year") \
       .mode("overwrite") \
       .parquet(gold_path)


# ==========================================
# CÂU 3: TRUY VẤN TỪ DATA LAKE (BÁO CÁO NHANH)
# ==========================================
print("✅ 3. Hoàn tất Pipeline! Đang lấy báo cáo năm 2011...")
# Yêu cầu: Đọc thẳng vào thư mục phân vùng Năm 2011 của tầng Gold
# (Gợi ý: Dùng đường dẫn gold_path cộng thêm thư mục phân vùng)
df_report_2011 = spark.read.parquet(f"{gold_path}/Year=2011")

# Hiển thị Top 5 quốc gia có doanh thu cao nhất năm 2011
df_report_2011.orderBy(col("TotalRevenue").desc()).show(5)

Mounted at /content/drive
⏳ 1. Đang xử lý tầng Silver...
⏳ 2. Đang xử lý tầng Gold...
✅ 3. Hoàn tất Pipeline! Đang lấy báo cáo năm 2011...
+--------------+------------------+
|       Country|      TotalRevenue|
+--------------+------------------+
|United Kingdom| 6809729.704000123|
|   Netherlands| 276661.8599999993|
|          EIRE| 256732.0199999992|
|       Germany|213626.00000000023|
|        France|199407.74000000017|
+--------------+------------------+
only showing top 5 rows
